# Notebook 05 — Dynamic Learning & Feedback Loop

**Day 5 deliverable.** Demonstrates incremental learning with `partial_fit`, produces the killer demo visual (accuracy climbing as batches are fed), initialises the feedback store, simulates outcome feedback, and shows accuracy before vs after the update.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from src.feedback import init_db, log_feedback, load_feedback, retrain_on_feedback
import warnings
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
sns.set_theme(style='whitegrid')

In [ ]:
train_df     = pd.read_csv(PROC / 'train_clean.csv')
test_df      = pd.read_csv(PROC / 'test_clean.csv')
feature_cols = joblib.load(PROC / 'feature_cols.pkl')
le           = joblib.load(PROC / 'label_encoder.pkl')

X_train = train_df[feature_cols].values
y_train = le.transform(train_df['prognosis'])
X_test  = test_df[feature_cols].values
y_test  = le.transform(test_df['prognosis'])

n_classes = len(le.classes_)
print(f'Classes: {n_classes}  |  Train: {len(X_train)}  |  Test: {len(X_test)}')

## Incremental Learning Curve

We simulate streaming data by training an `SGDClassifier` one mini-batch at a time and recording test metrics after each batch. This chart is the visual proof that **the model gets smarter as more data flows in**.

In [ ]:
BATCH_SIZE  = 100
n_batches   = len(X_train) // BATCH_SIZE
all_classes = np.arange(n_classes)

online_model = SGDClassifier(loss='log_loss', random_state=42)

acc_history  = []
f1_history   = []
samples_seen = []

for i in range(n_batches):
    s = i * BATCH_SIZE
    online_model.partial_fit(X_train[s:s+BATCH_SIZE], y_train[s:s+BATCH_SIZE],
                             classes=all_classes)
    y_pred = online_model.predict(X_test)
    acc_history.append(accuracy_score(y_test, y_pred))
    f1_history.append(f1_score(y_test, y_pred, average='macro', zero_division=0))
    samples_seen.append(s + BATCH_SIZE)

print(f'After {n_batches} batches ({n_batches*BATCH_SIZE} samples):')
print(f'  Test accuracy : {acc_history[-1]:.4f}')
print(f'  Macro-F1      : {f1_history[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(samples_seen, acc_history, color='royalblue', linewidth=2)
axes[0].set_title('Top-1 Accuracy vs Samples Seen', fontsize=13)
axes[0].set_xlabel('Training Samples Seen')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_ylim(0, 1.05)

axes[1].plot(samples_seen, f1_history, color='darkorange', linewidth=2)
axes[1].set_title('Macro-F1 vs Samples Seen', fontsize=13)
axes[1].set_xlabel('Training Samples Seen')
axes[1].set_ylabel('Macro-F1')
axes[1].set_ylim(0, 1.05)

for ax in axes:
    ax.grid(True, alpha=0.4)

fig.suptitle('Incremental Learning Curve — Model Improves as More Data Flows In',
             fontsize=14)
plt.tight_layout()
plt.savefig(PROC / 'plot_incremental_learning_curve.png', dpi=150)
plt.show()

## Feedback Store Initialisation

In [ ]:
init_db()
print(f'Feedback store ready at: {ROOT / "data" / "feedback.db"}')

## Simulate Outcome Feedback

In production, feedback comes from the Streamlit app. Here we draw 20 test-set cases and log them as 'improved' to demonstrate the `partial_fit` update.

In [ ]:
ensemble_model = joblib.load(PROC / 'ensemble_model.pkl')
acc_before     = accuracy_score(y_test, ensemble_model.predict(X_test))
print(f'Ensemble accuracy BEFORE feedback: {acc_before:.4f}')

np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=20, replace=False)

for idx in sample_idx:
    vec       = X_test[idx]
    symptoms  = [feature_cols[j] for j in range(len(feature_cols)) if vec[j] == 1]
    probs     = ensemble_model.predict_proba(vec.reshape(1, -1))[0]
    top_idx   = np.argsort(probs)[::-1][:3]
    top_k     = [(le.inverse_transform([i])[0], round(float(probs[i]), 4)) for i in top_idx]
    log_feedback(
        symptoms=symptoms,
        top_conditions=top_k,
        recommended_treatments={d: [] for d, _ in top_k},
        outcome='improved',
    )

fb_df = load_feedback()
print(f'Feedback records logged: {len(fb_df)}')
fb_df.head(3)

## Partial-Fit Update & Before/After Comparison

In [ ]:
online_after, msg = retrain_on_feedback()
print(msg)

if online_after is not None:
    acc_after = accuracy_score(y_test, online_after.predict(X_test))
    print(f'Online model accuracy AFTER  feedback: {acc_after:.4f}')
    print(f'Change: {acc_after - acc_before:+.4f}')

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(
        ['Ensemble\n(before feedback)', 'Online model\n(after feedback)'],
        [acc_before, acc_after],
        color=['steelblue', 'darkorange'],
        edgecolor='white', width=0.5,
    )
    ax.bar_label(bars, fmt='%.4f', padding=3)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Test Accuracy')
    ax.set_title('Accuracy Before vs After Feedback Update', fontsize=13)
    plt.tight_layout()
    plt.savefig(PROC / 'plot_before_after_feedback.png', dpi=150)
    plt.show()

## Summary

| Artefact | Location |
|---|---|
| Incremental learning curve | `data/processed/plot_incremental_learning_curve.png` |
| Before/after feedback chart | `data/processed/plot_before_after_feedback.png` |
| Online model (partial-fit) | `data/processed/online_model.pkl` |
| Feedback store | `data/feedback.db` |

Pipeline complete. Run the Streamlit app with:
```
streamlit run app/app.py
```